# OralSkop — serve the trained model as an API

Wraps a trained **torchseg** checkpoint (the custom semantic-seg path) in a FastAPI
server, runs it on a background thread so this notebook stays interactive, and opens a
public **ngrok** URL you can hand to a friend.

**Run this notebook from the `ai/` directory** (so `runs/...` resolves). Install the
serving deps once:

```bash
uv sync --extra serve
```

Get a free ngrok authtoken at https://dashboard.ngrok.com/get-started/your-authtoken .

In [ ]:
from oralskop.serve.notebook import serve

server = serve(
    weights="runs/seg/deeplabv3_alphadent/best.pt",  # path to YOUR trained checkpoint
    # arch="deeplabv3_resnet50",   # only needed if the checkpoint has no metadata
    imgsz=512,
    device="cuda",                 # use "cpu" if no GPU
    conf=0.5,                       # min mean-softmax confidence per detection
    ngrok_authtoken="PASTE_YOUR_NGROK_TOKEN_HERE",
)
print("\nShare this URL:", server.url)

## What your friend can do

- Open **`<url>/`** in a browser, pick a photo, hit *Analyze* — sees the overlay + JSON.
- Or POST an image programmatically:
  ```bash
  curl -F file=@photo.jpg <url>/predict            # JSON detections
  curl -F file=@photo.jpg <url>/predict/overlay -o out.png   # annotated image
  ```

Endpoints: `GET /` (form), `GET /health`, `GET /info`, `POST /predict`, `POST /predict/overlay`.

In [ ]:
# Quick local self-test: send an image and show the returned overlay inline.
import io, requests
from PIL import Image

test_image = "data/alphadent/images/val"  # a folder of built val images, or any .jpg path
import os, glob
if os.path.isdir(test_image):
    test_image = sorted(glob.glob(os.path.join(test_image, "*")))[0]
print("Testing on:", test_image)

with open(test_image, "rb") as f:
    img_bytes = f.read()

print(requests.post(f"{server.local_url}/predict", files={"file": img_bytes}).json())
overlay = requests.post(f"{server.local_url}/predict/overlay", files={"file": img_bytes}).content
Image.open(io.BytesIO(overlay))

In [ ]:
# Shut the server (and ngrok tunnel) down when you're done.
server.stop()